# Analyse av utgiftskrav

Denne notatboken viser hvordan man lager agenter som bruker plugins for å behandle reiseutgifter fra lokale kvitteringsbilder, generere en utgiftskravs-e-post og visualisere utgiftsdata ved hjelp av et sektordiagram. Agenter velger dynamisk funksjoner basert på oppgavens kontekst.

Trinn:
1. OCR-agent behandler det lokale kvitteringsbildet og henter ut reiseutgiftsdata.
2. E-postagent genererer en utgiftskravs-e-post.

### Eksempel på et reisekostnadsscenario:
Tenk deg at du er en ansatt som reiser på en forretningsreise til en annen by. Selskapet ditt har en policy om å refundere alle rimelige reisekostnader. Her er en oversikt over potensielle reiseutgifter:
- Transport:
Flybillett for tur-retur fra hjemstedet ditt til destinasjonsbyen.
Taxi eller skyss-tjenester til og fra flyplassen.
Lokal transport i destinasjonsbyen (som kollektivtransport, leiebil eller taxi).

- Overnatting:
Hotellopphold i tre netter på et middels stort forretningshotell nær møteplassen.

- Måltider:
Daglig måltidstilskudd for frokost, lunsj og middag, basert på selskapets diettpolicy.

- Diverse utgifter:
Parkeringsavgift på flyplassen.
Internettkostnader på hotellet.
Tips eller små serviceavgifter.

- Dokumentasjon:
Du leverer alle kvitteringer (fly, taxi, hotell, måltider osv.) og en fullført utgiftsrapport for refusjon.


## Importer nødvendige biblioteker

Importer de nødvendige bibliotekene og modulene for notatblokken.


In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import base64
import dotenv
from typing import Annotated, List

from pydantic import BaseModel, Field

from agent_framework import tool, AgentResponseUpdate, WorkflowBuilder
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

 ## Definer utgiftsmodeller

 Lag en Pydantic-modell for individuelle utgifter og en ExpenseFormatter-klasse for å konvertere en brukerspørsmål til strukturert utgiftsdata.

 Hver utgift vil bli representert i formatet:
 `{'date': '07-Mar-2025', 'description': 'flight to destination', 'amount': 675.99, 'category': 'Transportation'}`


In [ ]:
class Expense(BaseModel):
    date: str = Field(..., description="Date of expense in dd-MMM-yyyy format")
    description: str = Field(..., description="Expense description")
    amount: float = Field(..., description="Expense amount")
    category: str = Field(..., description="Expense category (e.g., Transportation, Meals, Accommodation, Miscellaneous)")

class ExpenseFormatter(BaseModel):
    raw_query: str = Field(..., description="Raw query input containing expense details")
    
    def parse_expenses(self) -> List[Expense]:
        """
        Parses the raw query into a list of Expense objects.
        Expected format: "date|description|amount|category" separated by semicolons.
        """
        expense_list = []
        for expense_str in self.raw_query.split(";"):
            if expense_str.strip():
                parts = expense_str.strip().split("|")
                if len(parts) == 4:
                    date, description, amount, category = parts
                    try:
                        expense = Expense(
                            date=date.strip(),
                            description=description.strip(),
                            amount=float(amount.strip()),
                            category=category.strip()
                        )
                        expense_list.append(expense)
                    except ValueError as e:
                        print(f"[LOG] Parse Error: Invalid data in '{expense_str}': {e}")
        return expense_list

## Definere verktøy - Generere e-posten

Lag en verktøyfunksjon for å generere en e-post for innsending av et utgiftskrav.
- Dette verktøyet bruker `@tool`-dekoratøren fra Microsoft Agent Framework.
- Det beregner totalbeløpet for utgiftene og formaterer detaljene til en e-posttekst.


In [ ]:
@tool(approval_mode="never_require")
def generate_expense_email(
    expense_data: Annotated[str, "Semicolon-separated expense entries in 'date|description|amount|category' format"]
) -> str:
    """Generate an email to submit an expense claim to the Finance Team."""
    formatter = ExpenseFormatter(raw_query=expense_data)
    expenses = formatter.parse_expenses()
    if not expenses:
        return "No valid expenses found to include in the email."
    total_amount = sum(e.amount for e in expenses)
    email_body = "Dear Finance Team,\n\n"
    email_body += "Please find below the details of my expense claim:\n\n"
    for e in expenses:
        email_body += f"- {e.date} | {e.description}: ${e.amount:.2f} ({e.category})\n"
    email_body += f"\nTotal Amount: ${total_amount:.2f}\n\n"
    email_body += "Receipts for all expenses are attached for your reference.\n\n"
    email_body += "Thank you,\n[Your Name]"
    return email_body

# Tool for Extracting Travel Expenses from Receipt Images

Create a tool function to extract travel expenses from receipt images.
- This tool uses the `@tool` decorator from the Microsoft Agent Framework.
- It reads the receipt image, encodes it as base64, and returns the data URI for the agent to analyze.


In [ ]:
@tool(approval_mode="never_require")
def load_receipt_image(
    image_path: Annotated[str, "Path to the receipt image file"] = "receipt.jpg"
) -> str:
    """Load a receipt image and return its base64-encoded data URI for OCR extraction."""
    try:
        with open(image_path, "rb") as f:
            image_data = base64.b64encode(f.read()).decode("utf-8")
        return f"data:image/jpeg;base64,{image_data}"
    except Exception as e:
        error_msg = f"[LOG] Error loading image '{image_path}': {str(e)}"
        print(error_msg)
        return error_msg

## Behandle utgifter

Definer agentene og koble dem sammen i en sekvensiell arbeidsflyt ved hjelp av `WorkflowBuilder`.
- OCR-agenten henter ut strukturert utgiftsdata fra kvitteringsbildet ved hjelp av `load_receipt_image`-verktøyet.
- E-postagenten tar de uttrukne dataene og genererer en profesjonell utgiftskrav-epost ved hjelp av `generate_expense_email`-verktøyet.
- `WorkflowBuilder` med `add_edge` lager en sekvensiell pipeline: OCR-agent → E-postagent.


In [ ]:
ocr_agent = client.as_agent(
    tools=[load_receipt_image],
    name="OCRAgent",
    instructions=(
        "You are an expert OCR assistant specialized in extracting structured data from receipt images. "
        "Use the 'load_receipt_image' tool to load the receipt image, then analyze it and extract "
        "travel-related expense details in the format: 'date|description|amount|category' separated by semicolons. "
        "Follow these rules: "
        "- Date: Convert dates (e.g., '4/4/22') to 'dd-MMM-yyyy' (e.g., '04-Apr-2022'). "
        "- Description: Extract item names. "
        "- Amount: Use numeric values (e.g., '4.50' from '$4.50'). "
        "- Category: Infer from context (e.g., 'Meals' for food, 'Transportation' for travel, "
        "'Accommodation' for lodging, 'Miscellaneous' otherwise). "
        "Ignore totals, subtotals, or service charges unless they are itemized expenses. "
        "If no expenses are found, return 'No expenses detected'. "
        "Return only the structured data, no additional text."
    ),
)

email_agent = client.as_agent(
    name="EmailAgent",
    tools=[generate_expense_email],
    instructions=(
        "You are an expense claim email generator. Take the travel expense data from the previous agent "
        "(in 'date|description|amount|category' format separated by semicolons) and use the "
        "'generate_expense_email' tool to produce a professional expense claim email. "
        "Pass the semicolon-separated expense data directly to the tool."
    ),
)

## Hovedfunksjon

Bygg den sekvensielle arbeidsflyten og kjør den for å behandle kvitteringsbildet og generere e-posten for reiseregningen.


> **Merk:** Denne arbeidsflyten sender for øyeblikket kvitteringsbildet som base64-tekst, noe de fleste chatmodeller (inkludert gpt-4o) ikke vil behandle som et bilde.
> Det kan også overskride modellens kontekstvindu. Foretrekk å kjøre OCR med Azure AI Vision (eller et annet OCR-verktøy) og bare sende den uttrukne teksten, eller refaktorer for å sende bildet som en `image_url`-melding.
> Hvis du bare vil unngå kontekstfeil, prøv et mindre kvitteringsbilde eller en modell med et større kontekstvindu.


In [ ]:
workflow = WorkflowBuilder(start_executor=ocr_agent) \
    .add_edge(ocr_agent, email_agent) \
    .build()

prompt = (
    "Please extract the raw text from the receipt image at 'receipt.jpg', "
    "focusing on travel expenses like dates, descriptions, amounts, and categories "
    "(e.g., Transportation, Accommodation, Meals, Miscellaneous). "
    "Then generate a professional expense claim email."
)

last_author = None
events = workflow.run(
    prompt,
    stream=True,
)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"# Agent - {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Ansvarsfraskrivelse**:
Dette dokumentet er oversatt ved hjelp av AI-oversettelsestjenesten [Co-op Translator](https://github.com/Azure/co-op-translator). Selv om vi streber etter nøyaktighet, vær oppmerksom på at automatiske oversettelser kan inneholde feil eller unøyaktigheter. Det opprinnelige dokumentet på originalspråket skal betraktes som den autoritative kilden. For kritisk informasjon anbefales profesjonell menneskelig oversettelse. Vi er ikke ansvarlige for eventuelle misforståelser eller feiltolkninger som oppstår ved bruk av denne oversettelsen.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
